# NYC Taxi Operations Analysis

Exploratory analysis of 2023 NYC yellow taxi trips, including data cleaning, hourly sampling, demand patterns, fare analysis, and geographic visualisation.

This educational project includes starter material and exercises. See the README for the data requirements. Results should be reproduced with the appropriate dataset.


# **New York City Yellow Taxi Data**

## Objective
In this case study you will be learning exploratory data analysis (EDA) with the help of a dataset on yellow taxi rides in New York City. This will enable you to understand why EDA is an important step in the process of data science and machine learning.

## **Problem Statement**
As an analyst at an upcoming taxi operation in NYC, you are tasked to use the 2023 taxi trip data to uncover insights that could help optimise taxi operations. The goal is to analyse patterns in the data that can inform strategic decisions to improve service efficiency, maximise revenue, and enhance passenger experience.

## Tasks
You need to perform the following steps for successfully completing this assignment:
1. Data Loading
2. Data Cleaning
3. Exploratory Analysis: Bivariate and Multivariate
4. Creating Visualisations to Support the Analysis
5. Deriving Insights and Stating Conclusions

---

**NOTE:** The marks given along with headings and sub-headings are cumulative marks for those particular headings/sub-headings.<br>

The actual marks for each task are specified within the tasks themselves.

For example, marks given with heading *2* or sub-heading *2.1* are the cumulative marks, for your reference only. <br>

The marks you will receive for completing tasks are given with the tasks.

Suppose the marks for two tasks are: 3 marks for 2.1.1 and 2 marks for 3.2.2, or
* 2.1.1 [3 marks]
* 3.2.2 [2 marks]

then, you will earn 3 marks for completing task 2.1.1 and 2 marks for completing task 3.2.2.


---

## Data Understanding
The yellow taxi trip records include fields capturing pick-up and drop-off dates/times, pick-up and drop-off locations, trip distances, itemized fares, rate types, payment types, and driver-reported passenger counts.

The data is stored in Parquet format (*.parquet*). The dataset is from 2009 to 2024. However, for this assignment, we will only be using the data from 2023.

The data for each month is present in a different parquet file. You will get twelve files for each of the months in 2023.

The data was collected and provided to the NYC Taxi and Limousine Commission (TLC) by technology providers like vendors and taxi hailing apps. <br>

You can find the link to the TLC trip records page here: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

###  Data Description
You can find the data description here: [Data Dictionary](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf)

**Trip Records**



|Field Name       |description |
|:----------------|:-----------|
| VendorID | A code indicating the TPEP provider that provided the record. <br> 1= Creative Mobile Technologies, LLC; <br> 2= VeriFone Inc. |
| tpep_pickup_datetime | The date and time when the meter was engaged.  |
| tpep_dropoff_datetime | The date and time when the meter was disengaged.   |
| Passenger_count | The number of passengers in the vehicle. <br> This is a driver-entered value. |
| Trip_distance | The elapsed trip distance in miles reported by the taximeter. |
| PULocationID | TLC Taxi Zone in which the taximeter was engaged |
| DOLocationID | TLC Taxi Zone in which the taximeter was disengaged |
|RateCodeID |The final rate code in effect at the end of the trip.<br> 1 = Standard rate <br> 2 = JFK <br> 3 = Newark <br>4 = Nassau or Westchester <br>5 = Negotiated fare <br>6 = Group ride |
|Store_and_fwd_flag |This flag indicates whether the trip record was held in vehicle memory before sending to the vendor, aka “store and forward,” because the vehicle did not have a connection to the server.  <br>Y= store and forward trip <br>N= not a store and forward trip |
|Payment_type| A numeric code signifying how the passenger paid for the trip. <br> 1 = Credit card <br>2 = Cash <br>3 = No charge <br>4 = Dispute <br>5 = Unknown <br>6 = Voided trip |
|Fare_amount| The time-and-distance fare calculated by the meter. <br>Extra Miscellaneous extras and surcharges.  Currently, this only includes the 0.50 and 1 USD rush hour and overnight charges. |
|MTA_tax |0.50 USD MTA tax that is automatically triggered based on the metered rate in use. |
|Improvement_surcharge | 0.30 USD improvement surcharge assessed trips at the flag drop. The improvement surcharge began being levied in 2015. |
|Tip_amount |Tip amount – This field is automatically populated for credit card tips. Cash tips are not included. |
| Tolls_amount | Total amount of all tolls paid in trip.  |
| total_amount | The total amount charged to passengers. Does not include cash tips. |
|Congestion_Surcharge |Total amount collected in trip for NYS congestion surcharge. |
| Airport_fee | 1.25 USD for pick up only at LaGuardia and John F. Kennedy Airports|

Although the amounts of extra charges and taxes applied are specified in the data dictionary, you will see that some cases have different values of these charges in the actual data.

**Taxi Zones**

Each of the trip records contains a field corresponding to the location of the pickup or drop-off of the trip, populated by numbers ranging from 1-263.

These numbers correspond to taxi zones, which may be downloaded as a table or map/shapefile and matched to the trip records using a join.

This is covered in more detail in later sections.

---

## **1** Data Preparation

<font color = red>[5 marks]</font> <br>

### Import Libraries

In [ ]:
# Import warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Import the libraries you will be using for analysisimport numpy as np
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import pyarrow as pa
import pyarrow.parquet as pq
import glob

try:
    import geopandas as gpd
    from shapely.geometry import Point
except Exception as e:
    print("Geo libraries not loaded yet (ok for now). We'll install when needed.")

plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

In [ ]:
# Recommended versions
# numpy version: 1.26.4
# pandas version: 2.2.2
# matplotlib version: 3.10.0
# seaborn version: 0.13.2

# Check versions
print("numpy version:", np.__version__)
print("pandas version:", pd.__version__)
print("matplotlib version:", plt.matplotlib.__version__)
print("seaborn version:", sns.__version__)

In [ ]:
from pathlib import Path
BASE_PATH = str(Path("data").resolve())


### **1.1** Load the dataset
<font color = red>[5 marks]</font> <br>

You will see twelve files, one for each month.

To read parquet files with Pandas, you have to follow a similar syntax as that for CSV files.

`df = pd.read_parquet('file.parquet')`

In [ ]:
# Try loading one file

# df = pd.read_parquet('2023-1.parquet')
df = pd.read_parquet(f"{BASE_PATH}/trip_records/2023-1.parquet")
df.info()

How many rows are there? Do you think handling such a large number of rows is computationally feasible when we have to combine the data for all twelve months into one?

Answer: The January 2023 yellow taxi dataset contains approximately 3.6 million trip records. Combining all twelve months would result in over 40 million records, which would be computationally expensive and memory-intensive in a Google Colab environment. Therefore, directly loading and processing the full dataset is not feasible, and sampling is required to efficiently analyze the data while preserving overall trends.

To handle this, we need to sample a fraction of data from each of the files. How to go about that? Think of a way to select only some portion of the data from each month's file that accurately represents the trends.

Answer: To handle this, a sampling strategy can be applied to each monthly file before combining them. A small fraction of trips (for example, 5%) is randomly selected for every hour of every date using the tpep_pickup_datetime field. This ensures that hourly demand patterns, daily variations such as weekdays and weekends, and seasonal trends across months are preserved, while significantly reducing the overall data size and making the analysis computationally feasible.

#### Sampling the Data
> One way is to take a small percentage of entries for pickup in every hour of a date. So, for all the days in a month, we can iterate through the hours and select 5% values randomly from those. Use `tpep_pickup_datetime` for this. Separate date and hour from the datetime values and then for each date, select some fraction of trips for each of the 24 hours.

To sample data, you can use the `sample()` method. Follow this syntax:

```Python
# sampled_data is an empty DF to keep appending sampled data of each hour
# hour_data is the DF of entries for an hour 'X' on a date 'Y'

sample = hour_data.sample(frac = 0.05, random_state = 42)
# sample 0.05 of the hour_data
# random_state is just a seed for sampling, you can define it yourself

sampled_data = pd.concat([sampled_data, sample]) # adding data for this hour to the DF
```

This *sampled_data* will contain 5% values selected at random from each hour.

Note that the code given above is only the part that will be used for sampling and not the complete code required for sampling and combining the data files.

Keep in mind that you sample by date AND hour, not just hour. (Why?)

---

**1.1.1** <font color = red>[5 marks]</font> <br>
Figure out how to sample and combine the files.

**Note:** It is not mandatory to use the method specified above. While sampling, you only need to make sure that your sampled data represents the overall data of all the months accurately.

In [ ]:
# Sample the data
# It is recommmended to not load all the files at once to avoid memory overload

In [ ]:
# Take a small percentage of entries from each hour of every date.
# Iterating through the monthly data:
#   read a month file -> day -> hour: append sampled data -> move to next hour -> move to next day after 24 hours -> move to next month file
# Create a single dataframe for the year combining all the monthly data

# Select the folder having data files
import os

# Select the folder having data files
os.chdir(f"{BASE_PATH}/trip_records")

# Create a list of all the twelve files to read
#file_list = os.listdir()
file_list = [f for f in os.listdir() if f.endswith(".parquet") and f.startswith("2023-")]
file_list = sorted(
    file_list,
    key=lambda x: int(x.split("-")[1].split(".")[0])
)
print("Files found:", len(file_list))
print("First 3 files:", file_list[:3])

# initialise an empty dataframe
df = pd.DataFrame()


# iterate through the list of files and sample one by one:
for file_name in file_list:
    try:
        # file path for the current file
        file_path = os.path.join(os.getcwd(), file_name)

        # Reading the current file
        month_df = pd.read_parquet(file_path)

        # Convert pickup datetime (needed for date/hour sampling)
        month_df["tpep_pickup_datetime"] = pd.to_datetime(
            month_df["tpep_pickup_datetime"], errors="coerce"
        )

        # Create date and hour columns
        month_df["pickup_date"] = month_df["tpep_pickup_datetime"].dt.date
        month_df["pickup_hour"] = month_df["tpep_pickup_datetime"].dt.hour


        # We will store the sampled data for the current date in this df by appending the sampled data from each hour to this
        # After completing iteration through each date, we will append this data to the final dataframe.
        sampled_data = pd.DataFrame()

        # Loop through dates and then loop through every hour of each date
        for date in month_df["pickup_date"].dropna().unique():
            # data for a single date
            date_data = month_df[month_df["pickup_date"] == date]

            # Iterate through each hour of the selected date
            for hour, hour_data in date_data.groupby("pickup_hour"):
                if len(hour_data) > 0:
                    sample = hour_data.sample(frac=0.05, random_state=42)
                    sampled_data = pd.concat([sampled_data, sample], ignore_index=True)

                # Sample 5% of the hourly data randomly
                if len(hour_data) > 0:
                    sample = hour_data.sample(frac=0.05, random_state=42)

                # add data of this hour to the dataframe
                sampled_data = pd.concat([sampled_data, sample], ignore_index=True)

        # Concatenate the sampled data of all the dates to a single dataframe
        df = pd.concat([df, sampled_data], ignore_index=True) # we initialised this empty DF earlier

        # Free memory
        del month_df
        print(f"Done: {file_name} | sampled rows: {sampled_data.shape[0]}")

    except Exception as e:
        print(f"Error reading file {file_name}: {e}")
print("Final combined sampled df shape:", df.shape)

After combining the data files into one DataFrame, convert the new DataFrame to a CSV or parquet file and store it to use directly.

Ideally, you can try keeping the total entries to around 250,000 to 300,000.

In [ ]:
# Store the df in csv/parquet
# df.to_parquet('')
df = df.sample(n=300_000, random_state=42).reset_index(drop=True)

df.shape

In [ ]:
# Save sampled dataset for reuse
df.to_parquet(f"{BASE_PATH}/yellow_2023_sampled.parquet", index=False)

print("Saved sampled dataset to Drive.")

## **2** Data Cleaning
<font color = red>[30 marks]</font> <br>

Now we can load the new data directly.

In [ ]:
# Load the new data file
df = pd.read_parquet(f"{BASE_PATH}/yellow_2023_sampled.parquet")

In [ ]:
df.head()

In [ ]:
df.info()

#### **2.1** Fixing Columns
<font color = red>[10 marks]</font> <br>

Fix/drop any columns as you seem necessary in the below sections

**2.1.1** <font color = red>[2 marks]</font> <br>

Fix the index and drop unnecessary columns

In [ ]:
# Fix the index and drop any columns that are not needed
df.reset_index(drop=True, inplace=True)

df.drop(columns=["pickup_date", "pickup_hour"], inplace=True)
df.info()

**2.1.2** <font color = red>[3 marks]</font> <br>
There are two airport fee columns. This is possibly an error in naming columns. Let's see whether these can be combined into a single column.

Answer: Yes

In [ ]:
# Combine the two airport fee columns
df["airport_fee"] = df[["airport_fee", "Airport_fee"]].max(axis=1)

# Drop the duplicate column
df.drop(columns=["Airport_fee"], inplace=True)

In [ ]:
df[["airport_fee"]].describe()

**2.1.3** <font color = red>[5 marks]</font> <br>
Fix columns with negative (monetary) values

In [ ]:
# check where values of fare amount are negative
df[df["fare_amount"] < 0].head()
#There are zero rows where fare_amount is negative.

In [ ]:
(df["fare_amount"] < 0).sum()
#There are zero rows where fare_amount is negative.

Did you notice something different in the `RatecodeID` column for above records?

In [ ]:
# Analyse RatecodeID for the negative fare amounts
df.loc[df["fare_amount"] < 0, "RatecodeID"].value_counts(dropna=False)
#No difference was observed in the RatecodeID column because there are no records with negative fare_amount values in the sampled dataset. As a result, no specific RatecodeID pattern is associated with negative fares.

In [ ]:
# Find which columns have negative values
numeric_cols = df.select_dtypes(include=["number"]).columns

negatives_summary = {
    col: (df[col] < 0).sum()
    for col in numeric_cols
}

# Convert to DataFrame for readability
negatives_df = (
    pd.DataFrame.from_dict(negatives_summary, orient="index", columns=["negative_count"])
    .sort_values("negative_count", ascending=False)
)

negatives_df

In [ ]:
# fix these negative values
# Monetary columns should not be negative in a real-world scenario.
# We cap negative values at 0 to handle reversals or data-entry errors.

monetary_cols = [
    "total_amount",
    "improvement_surcharge",
    "mta_tax",
    "congestion_surcharge",
    "airport_fee"
]

for col in monetary_cols:
    if col in df.columns:
        df[col] = df[col].clip(lower=0)

In [ ]:
(df[monetary_cols] < 0).sum()

### **2.2** Handling Missing Values
<font color = red>[10 marks]</font> <br>

**2.2.1**  <font color = red>[2 marks]</font> <br>
Find the proportion of missing values in each column




In [ ]:
# Find the proportion of missing values in each column
missing_proportion = df.isnull().mean().sort_values(ascending=False)

missing_proportion

**2.2.2**  <font color = red>[3 marks]</font> <br>
Handling missing values in `passenger_count`

In [ ]:
# Display the rows with null values
# Impute NaN values in 'passenger_count'
df[df["passenger_count"].isnull()].head()

In [ ]:
(df["passenger_count"] == 0).sum()

In [ ]:
# Treat passenger_count = 0 as missing
df.loc[df["passenger_count"] == 0, "passenger_count"] = np.nan
# Impute missing passenger_count using the median
median_passenger_count = df["passenger_count"].median()
df["passenger_count"].fillna(median_passenger_count, inplace=True)

Did you find zeroes in passenger_count? Handle these.

Answer: No zero values were found in the passenger_count column in the sampled dataset. However, as a precautionary step, any potential zero values (which would indicate unrecorded passenger information rather than actual zero-passenger trips) were treated as missing and handled consistently with other missing values. Missing passenger counts were imputed using the median passenger count to preserve realistic trip characteristics.

**2.2.3**  <font color = red>[2 marks]</font> <br>
Handle missing values in `RatecodeID`

In [ ]:
# Fix missing values in 'RatecodeID'
# Check missing values in RatecodeID
df["RatecodeID"].isnull().sum()

In [ ]:
# Impute missing RatecodeID using the most frequent value (mode)
ratecode_mode = df["RatecodeID"].mode()[0]
df["RatecodeID"].fillna(ratecode_mode, inplace=True)

In [ ]:
# Verify no missing values remain
df["RatecodeID"].isnull().sum()

**2.2.4**  <font color = red>[3 marks]</font> <br>
Impute NaN in `congestion_surcharge`

In [ ]:
# handle null values in congestion_surcharge
# Check missing values
df["congestion_surcharge"].isnull().sum()

In [ ]:
# Impute missing congestion_surcharge with 0
df["congestion_surcharge"].fillna(0, inplace=True)

In [ ]:
# Verify no missing values remain
df["congestion_surcharge"].isnull().sum()

Are there missing values in other columns? Did you find NaN values in some other set of columns? Handle those missing values below.

In [ ]:
# Handle any remaining missing values
# Check remaining missing values
df.isnull().sum().sort_values(ascending=False)

In [ ]:
# Impute missing values in store_and_fwd_flag
df["store_and_fwd_flag"].fillna("N", inplace=True)

# Impute missing values in airport_fee
df["airport_fee"].fillna(0, inplace=True)

# Final check
df.isnull().sum()

### **2.3** Handling Outliers
<font color = red>[10 marks]</font> <br>

Before we start fixing outliers, let's perform outlier analysis.

In [ ]:
# Describe the data and check if there are any potential outliers present
# Check for potential out of place values in various columns
df.describe()

In [ ]:
# Select key numerical columns for outlier inspection
numeric_cols = [
    "trip_distance",
    "fare_amount",
    "total_amount",
    "tip_amount",
    "tolls_amount"
]

# Visualize boxplots to identify outliers
plt.figure(figsize=(12, 6))
df[numeric_cols].boxplot()
plt.xticks(rotation=45)
plt.title("Boxplot of Key Numerical Variables")
plt.show()

#yes — outliers exist, and boxplot shows them compressed near zero with huge spikes.

**2.3.1**  <font color = red>[10 marks]</font> <br>
Based on the above analysis, it seems that some of the outliers are present due to errors in registering the trips. Fix the outliers.

Some points you can look for:
- Entries where `trip_distance` is nearly 0 and `fare_amount` is more than 300
- Entries where `trip_distance` and `fare_amount` are 0 but the pickup and dropoff zones are different (both distance and fare should not be zero for different zones)
- Entries where `trip_distance` is more than 250  miles.
- Entries where `payment_type` is 0 (there is no payment_type 0 defined in the data dictionary)

These are just some suggestions. You can handle outliers in any way you wish, using the insights from above outlier analysis.

How will you fix each of these values? Which ones will you drop and which ones will you replace?

First, let us remove 7+ passenger counts as there are very less instances.

In [ ]:
# remove passenger_count > 6
df = df[df["passenger_count"] <= 6]

In [ ]:
# Continue with outlier handling
# Remove trips with near-zero distance but extremely high fare
df = df[~((df["trip_distance"] < 0.1) & (df["fare_amount"] > 300))]

# Remove trips with zero distance and zero fare but different pickup/dropoff zones
df = df[~(
    (df["trip_distance"] == 0) &
    (df["fare_amount"] == 0) &
    (df["PULocationID"] != df["DOLocationID"])
)]

# Remove unrealistically long trips
df = df[df["trip_distance"] <= 250]

# Handle undefined payment type (0)
df.loc[df["payment_type"] == 0, "payment_type"] = np.nan
df["payment_type"].fillna(df["payment_type"].mode()[0], inplace=True)

# Cap extreme monetary values
df["fare_amount"] = df["fare_amount"].clip(upper=300)
df["total_amount"] = df["total_amount"].clip(upper=400)
df["tip_amount"] = df["tip_amount"].clip(upper=100)
df["tolls_amount"] = df["tolls_amount"].clip(upper=100)

In [ ]:
# Do any columns need standardising?
# Standardise store_and_fwd_flag
# Valid values are 'Y' and 'N'; treat missing as 'N'
df["store_and_fwd_flag"] = df["store_and_fwd_flag"].fillna("N")

# Standardise passenger_count to integer after cleaning
df["passenger_count"] = df["passenger_count"].astype(int)

## **3** Exploratory Data Analysis
<font color = red>[90 marks]</font> <br>

In [ ]:
df.columns.tolist()

#### **3.1** General EDA: Finding Patterns and Trends
<font color = red>[40 marks]</font> <br>

**3.1.1** <font color = red>[3 marks]</font> <br>
Categorise the varaibles into Numerical or Categorical.
* `VendorID`:
* `tpep_pickup_datetime`:
* `tpep_dropoff_datetime`:
* `passenger_count`:
* `trip_distance`:
* `RatecodeID`:
* `PULocationID`:
* `DOLocationID`:
* `payment_type`:
* `pickup_hour`:
* `trip_duration`:


The following monetary parameters belong in the same category, is it categorical or numerical?


* `fare_amount`
* `extra`
* `mta_tax`
* `tip_amount`
* `tolls_amount`
* `improvement_surcharge`
* `total_amount`
* `congestion_surcharge`
* `airport_fee`

##### Temporal Analysis

**3.1.2** <font color = red>[5 marks]</font> <br>
Analyse the distribution of taxi pickups by hours, days of the week, and months.

In [ ]:
df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour

In [ ]:
# Find and show the hourly trends in taxi pickups
# Hourly pickup distribution
hourly_pickups = df["pickup_hour"].value_counts().sort_index()

plt.figure(figsize=(10, 5))
hourly_pickups.plot(kind="bar")
plt.xlabel("Hour of Day")
plt.ylabel("Number of Pickups")
plt.title("Hourly Distribution of Taxi Pickups")
plt.show()

In [ ]:
# Find and show the daily trends in taxi pickups (days of the week)
# Create day of week from pickup datetime
df["pickup_day"] = df["tpep_pickup_datetime"].dt.day_name()

# Order days properly
day_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

daily_pickups = df["pickup_day"].value_counts().reindex(day_order)

plt.figure(figsize=(10, 5))
daily_pickups.plot(kind="bar")
plt.xlabel("Day of Week")
plt.ylabel("Number of Pickups")
plt.title("Daily Distribution of Taxi Pickups")
plt.show()

In [ ]:
# Show the monthly trends in pickups

# Create month from pickup datetime
df["pickup_month"] = df["tpep_pickup_datetime"].dt.month_name()

# Order months properly
month_order = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]

monthly_pickups = df["pickup_month"].value_counts().reindex(month_order)

plt.figure(figsize=(10, 5))
monthly_pickups.plot(kind="bar")
plt.xlabel("Month")
plt.ylabel("Number of Pickups")
plt.title("Monthly Distribution of Taxi Pickups")
plt.show()

##### Financial Analysis

Take a look at the financial parameters like `fare_amount`, `tip_amount`, `total_amount`, and also `trip_distance`. Do these contain zero/negative values?

In [ ]:
# Analyse the above parameters

In [ ]:
# Check negative values in key financial and distance columns
financial_cols = ["fare_amount", "tip_amount", "total_amount", "trip_distance"]

negative_summary = {
    col: (df[col] < 0).sum()
    for col in financial_cols
}

pd.DataFrame.from_dict(
    negative_summary,
    orient="index",
    columns=["negative_count"]
)

In [ ]:
# Check zero values in key financial and distance columns
zero_summary = {
    col: (df[col] == 0).sum()
    for col in financial_cols
}

pd.DataFrame.from_dict(
    zero_summary,
    orient="index",
    columns=["zero_count"]
)

Do you think it is beneficial to create a copy DataFrame leaving out the zero values from these?

Answer: Yes — but selectively, not blindly.
It is beneficial to create filtered copies of the dataset depending on the analysis.
Zero values in fare_amount and total_amount likely indicate cancelled or erroneous trips and should be excluded for financial analysis.
However, zero values in tip_amount are valid (no-tip rides) and should be retained.
Zero trip_distance values may represent data issues and can be removed for distance-based analysis.

**3.1.3** <font color = red>[2 marks]</font> <br>
Filter out the zero values from the above columns.

**Note:** The distance might be 0 in cases where pickup and drop is in the same zone. Do you think it is suitable to drop such cases of zero distance?

Answer: Zero values in fare_amount and total_amount were removed as they likely represent cancelled or erroneous trips.
However, zero trip_distance values were retained when pickup and drop-off locations were the same, as these can represent valid short-distance or intra-zone trips.
This ensures data quality without removing legitimate observations.

In [ ]:
# Create a df with non zero entries for the selected parameters.
# For financial analysis, remove zero fare and zero total amount
df_non_zero = df[
    (df["fare_amount"] > 0) &
    (df["total_amount"] > 0)
]

# Keep trip_distance = 0 only if pickup and dropoff locations are the same
df_non_zero = df_non_zero[
    (df_non_zero["trip_distance"] > 0) |
    (df_non_zero["PULocationID"] == df_non_zero["DOLocationID"])
]

df_non_zero.shape


**3.1.4** <font color = red>[3 marks]</font> <br>
Analyse the monthly revenue (`total_amount`) trend

In [ ]:
# Group data by month and analyse monthly revenue
# Create month column
df_non_zero["pickup_month"] = df_non_zero["tpep_pickup_datetime"].dt.to_period("M")

# Aggregate total revenue per month
monthly_revenue = (
    df_non_zero
    .groupby("pickup_month")["total_amount"]
    .sum()
    .sort_index()
)

# Plot monthly revenue trend
plt.figure(figsize=(10, 5))
monthly_revenue.plot(kind="line", marker="o")
plt.xlabel("Month")
plt.ylabel("Total Revenue")
plt.title("Monthly Revenue Trend (Total Amount)")
plt.grid(True)
plt.show()

monthly_revenue


**3.1.5** <font color = red>[3 marks]</font> <br>
Show the proportion of each quarter of the year in the revenue

In [ ]:
# Calculate proportion of each quarter
# Create quarter column from pickup datetime
df_non_zero["pickup_quarter"] = df_non_zero["tpep_pickup_datetime"].dt.to_period("Q")

# Aggregate total revenue per quarter
quarterly_revenue = (
    df_non_zero
    .groupby("pickup_quarter")["total_amount"]
    .sum()
)

# Convert to proportion
quarterly_proportion = quarterly_revenue / quarterly_revenue.sum()

# Plot revenue proportion by quarter
plt.figure(figsize=(8, 5))
quarterly_proportion.plot(kind="bar")
plt.xlabel("Quarter")
plt.ylabel("Proportion of Total Revenue")
plt.title("Proportion of Revenue by Quarter (2023)")
plt.grid(axis="y")
plt.show()

quarterly_proportion

**3.1.6** <font color = red>[3 marks]</font> <br>
Visualise the relationship between `trip_distance` and `fare_amount`. Also find the correlation value for these two.

**Hint:** You can leave out the trips with trip_distance = 0

In [ ]:
# Show how trip fare is affected by distance
# Remove trips with zero distance (as suggested)
df_dist_fare = df_non_zero[df_non_zero["trip_distance"] > 0]

# Scatter plot: trip_distance vs fare_amount
plt.figure(figsize=(8, 5))
plt.scatter(
    df_dist_fare["trip_distance"],
    df_dist_fare["fare_amount"],
    alpha=0.3
)
plt.xlabel("Trip Distance (miles)")
plt.ylabel("Fare Amount ($)")
plt.title("Relationship between Trip Distance and Fare Amount")
plt.grid(True)
plt.show()

# Calculate correlation
correlation = df_dist_fare["trip_distance"].corr(df_dist_fare["fare_amount"])
correlation

**3.1.7** <font color = red>[5 marks]</font> <br>
Find and visualise the correlation between:
1. `fare_amount` and trip duration (pickup time to dropoff time)
2. `fare_amount` and `passenger_count`
3. `tip_amount` and `trip_distance`

In [ ]:
# Show relationship between fare and trip duration
# Create trip duration in minutes
df_non_zero["trip_duration"] = (
    df_non_zero["tpep_dropoff_datetime"] - df_non_zero["tpep_pickup_datetime"]
).dt.total_seconds() / 60

# Remove non-positive durations
df_non_zero = df_non_zero[df_non_zero["trip_duration"] > 0]

# Scatter plot: Fare vs Trip Duration
plt.figure(figsize=(8, 5))
plt.scatter(
    df_non_zero["trip_duration"],
    df_non_zero["fare_amount"],
    alpha=0.3
)
plt.xlabel("Trip Duration (minutes)")
plt.ylabel("Fare Amount ($)")
plt.title("Fare Amount vs Trip Duration")
plt.grid(True)
plt.show()

# Correlation
fare_duration_corr = df_non_zero["fare_amount"].corr(df_non_zero["trip_duration"])
fare_duration_corr


In [ ]:
# Show relationship between fare and number of passengers
# Scatter plot: Fare vs Passenger Count
plt.figure(figsize=(8, 5))
plt.scatter(
    df_non_zero["passenger_count"],
    df_non_zero["fare_amount"],
    alpha=0.3
)
plt.xlabel("Passenger Count")
plt.ylabel("Fare Amount ($)")
plt.title("Fare Amount vs Passenger Count")
plt.grid(True)
plt.show()

# Correlation
fare_passenger_corr = df_non_zero["fare_amount"].corr(df_non_zero["passenger_count"])
fare_passenger_corr

In [ ]:
# Show relationship between tip and trip distance
# Scatter plot: Tip vs Trip Distance
plt.figure(figsize=(8, 5))
plt.scatter(
    df_non_zero["trip_distance"],
    df_non_zero["tip_amount"],
    alpha=0.3
)
plt.xlabel("Trip Distance (miles)")
plt.ylabel("Tip Amount ($)")
plt.title("Tip Amount vs Trip Distance")
plt.grid(True)
plt.show()

# Correlation
tip_distance_corr = df_non_zero["tip_amount"].corr(df_non_zero["trip_distance"])
tip_distance_corr

**3.1.8** <font color = red>[3 marks]</font> <br>
Analyse the distribution of different payment types (`payment_type`)

In [ ]:
# Analyse the distribution of different payment types (payment_type).
payment_distribution = df_non_zero["payment_type"].value_counts().sort_index()

payment_distribution

In [ ]:
# Map payment type codes to labels
payment_labels = {
    1: "Credit Card",
    2: "Cash",
    3: "No Charge",
    4: "Dispute"
}

payment_distribution_labeled = payment_distribution.rename(index=payment_labels)

payment_distribution_labeled

In [ ]:
# Plot payment type distribution
plt.figure(figsize=(8, 5))
payment_distribution_labeled.plot(kind="bar")
plt.xlabel("Payment Type")
plt.ylabel("Number of Trips")
plt.title("Distribution of Payment Types")
plt.grid(axis="y")
plt.show()

- 1= Credit card
- 2= Cash
- 3= No charge
- 4= Dispute



##### Geographical Analysis

For this, you have to use the *taxi_zones.shp* file from the *taxi_zones* folder.

There would be multiple files inside the folder (such as *.shx, .sbx, .sbn* etc). You do not need to import/read any of the files other than the shapefile, *taxi_zones.shp*.

Do not change any folder structure - all the files need to be present inside the folder for it to work.

The folder structure should look like this:
```
Taxi Zones
|- taxi_zones.shp.xml
|- taxi_zones.prj
|- taxi_zones.sbn
|- taxi_zones.shp
|- taxi_zones.dbf
|- taxi_zones.shx
|- taxi_zones.sbx

 ```

 You only need to read the `taxi_zones.shp` file. The *shp* file will utilise the other files by itself.

We will use the *GeoPandas* library for geopgraphical analysis
```
import geopandas as gpd
```

More about geopandas and shapefiles: [About](https://geopandas.org/en/stable/about.html)


Reading the shapefile is very similar to *Pandas*. Use `gpd.read_file()` function to load the data (*taxi_zones.shp*) as a GeoDataFrame. Documentation: [Reading and Writing Files](https://geopandas.org/en/stable/docs/user_guide/io.html)

In [ ]:
# !pip install geopandas

**3.1.9** <font color = red>[2 marks]</font> <br>
Load the shapefile and display it.

In [ ]:
# import geopandas as gpd
# Import geopandas
import geopandas as gpd

# Read the shapefile using geopandas
#zones = # read the .shp file using gpd
zones = gpd.read_file(f"{BASE_PATH}/taxi_zones/taxi_zones.shp")
zones.head()

Now, if you look at the DataFrame created, you will see columns like: `OBJECTID`,`Shape_Leng`, `Shape_Area`, `zone`, `LocationID`, `borough`, `geometry`.
<br><br>

Now, the `locationID` here is also what we are using to mark pickup and drop zones in the trip records.

The geometric parameters like shape length, shape area and geometry are used to plot the zones on a map.

This can be easily done using the `plot()` method.

In [ ]:
# print(zones.info())
# zones.plot()
# Inspect zones GeoDataFrame
zones.info()

In [ ]:
# Plot taxi zones map
zones.plot(figsize=(10, 10), edgecolor="black")

Now, you have to merge the trip records and zones data using the location IDs.



**3.1.10** <font color = red>[3 marks]</font> <br>
Merge the zones data into trip data using the `locationID` and `PULocationID` columns.

In [ ]:
# Merge zones and trip records using locationID and PULocationID
df_with_zones = df_non_zero.merge(
    zones[["LocationID", "zone", "borough", "geometry"]],
    left_on="PULocationID",
    right_on="LocationID",
    how="left"
)

df_with_zones.head()


In [ ]:
df_with_zones[["PULocationID", "zone", "borough"]].sample(5)

**3.1.11** <font color = red>[3 marks]</font> <br>
Group data by location IDs to find the total number of trips per location ID

In [ ]:
# Group data by location and calculate the number of trips
trip_counts = (
    df_with_zones
    .groupby("PULocationID")
    .size()
    .reset_index(name="trip_count")
)

trip_counts.head()

**3.1.12** <font color = red>[2 marks]</font> <br>
Now, use the grouped data to add number of trips to the GeoDataFrame.

We will use this to plot a map of zones showing total trips per zone.

In [ ]:
# Merge trip counts back to the zones GeoDataFrame
zones_with_trips = zones.merge(
    trip_counts,
    left_on="LocationID",
    right_on="PULocationID",
    how="left"
)

# Replace NaN trip counts with 0 (zones with no pickups)
zones_with_trips["trip_count"] = zones_with_trips["trip_count"].fillna(0)

zones_with_trips.head()

The next step is creating a color map (choropleth map) showing zones by the number of trips taken.

Again, you can use the `zones.plot()` method for this. [Plot Method GPD](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.plot.html#geopandas.GeoDataFrame.plot)

But first, you need to define the figure and axis for the plot.

`fig, ax = plt.subplots(1, 1, figsize = (12, 10))`

This function creates a figure (fig) and a single subplot (ax)

---

After setting up the figure and axis, we can proceed to plot the GeoDataFrame on this axis. This is done in the next step where we use the plot method of the GeoDataFrame.

You can define the following parameters in the `zones.plot()` method:
```
column = '',
ax = ax,
legend = True,
legend_kwds = {'label': "label", 'orientation': "<horizontal/vertical>"}
```

To display the plot, use `plt.show()`.

**3.1.13** <font color = red>[3 marks]</font> <br>
Plot a color-coded map showing zone-wise trips

In [ ]:
# Define figure and axis
fig, ax = plt.subplots(1, 1, figsize=(12, 10))

# Plot the map and display it
zones_with_trips.plot(
    column="trip_count",
    ax=ax,
    legend=True
)

ax.set_title("NYC Taxi Pickups by Zone (Trip Count)")
ax.axis("off")
plt.show()

In [ ]:
zones_with_trips.plot(column="trip_count", ax=ax, legend=True, edgecolor="black", linewidth=0.2)

In [ ]:
# can you try displaying the zones DF sorted by the number of trips?
# Display zones sorted by number of trips (descending)
zones_sorted = zones_with_trips.sort_values(by="trip_count", ascending=False)

zones_sorted[["LocationID", "zone", "borough", "trip_count"]].head(10)

In [ ]:
# Top 10 zones with highest number of taxi trips
zones_sorted.tail(10)

Here we have completed the temporal, financial and geographical analysis on the trip records.

**Compile your findings from general analysis below:**

You can consider the following points:

* Busiest hours, days and months
Answer:
1. Taxi pickups are lowest during early morning hours (2–5 AM) and steadily increase through the day.
2. Peak demand occurs during evening hours, especially between 5 PM and 8 PM, likely driven by office commute and leisure travel.
3. Weekdays show higher pickup volumes than weekends, with Thursday and Friday being the busiest days.
4. Monthly analysis shows higher activity in late spring and fall, while February and August show relatively lower revenue and trip volumes.

* Trends in revenue collected
1. Monthly revenue closely follows pickup volume trends.
2. Revenue peaks in May and October, indicating seasonal travel and tourism effects.
3. Lower revenue is observed during mid-summer (August) and late winter (February).

* Trends in quarterly revenue
1. Q2 and Q4 contribute the highest proportion of annual revenue, together accounting for more than half of total revenue.
2. Q3 shows a noticeable dip, possibly due to vacation patterns and reduced business travel.
3. Revenue distribution across quarters is relatively balanced, indicating stable year-round demand with mild seasonality.

* How fare depends on trip distance, trip duration and passenger counts
* How tip amount depends on trip distance
1. Fare amount has a strong positive correlation with trip distance, confirming distance-based pricing.
2. Fare also shows a moderate positive correlation with trip duration, though duration alone is less predictive than distance.
3. Passenger count has very little impact on fare, indicating that fares are not passenger-dependent.
4. Tip amount increases with distance but shows higher variability, suggesting tipping behavior is influenced by factors beyond distance alone.
5. Credit cards dominate taxi payments, accounting for the vast majority of trips.
6. Cash payments form a much smaller share, while “No charge” and “Dispute” cases are rare, indicating clean transactional data.

* Busiest zones
1. The busiest pickup zones are concentrated in Manhattan and major transport hubs.
2. JFK Airport, LaGuardia Airport, Midtown, Upper East Side, and Times Square record the highest trip counts.
3. Peripheral zones such as parks, beaches, and low-density residential areas show very few or zero pickups, as expected.

Overall Conclusion

1. NYC taxi demand shows strong temporal and spatial patterns, with clear peak hours, business-day dominance, and location clustering.
2. Revenue is primarily driven by trip distance and volume, not passenger count.
3. The dataset is consistent, realistic, and suitable for further predictive modeling or policy analysis.

#### **3.2** Detailed EDA: Insights and Strategies
<font color = red>[50 marks]</font> <br>

Having performed basic analyses for finding trends and patterns, we will now move on to some detailed analysis focussed on operational efficiency, pricing strategies, and customer experience.

##### Operational Efficiency

Analyze variations by time of day and location to identify bottlenecks or inefficiencies in routes

**3.2.1** <font color = red>[3 marks]</font> <br>
Identify slow routes by calculating the average time taken by cabs to get from one zone to another at different hours of the day.

Speed on a route *X* for hour *Y* = (*distance of the route X / average trip duration for hour Y*)

In [ ]:
# Find routes which have the slowest speeds at different times of the day

# Create trip duration in hours
df_non_zero["trip_duration_hours"] = (
    (df_non_zero["tpep_dropoff_datetime"] - df_non_zero["tpep_pickup_datetime"])
    .dt.total_seconds() / 3600
)

# Remove invalid durations
df_non_zero = df_non_zero[df_non_zero["trip_duration_hours"] > 0]

# Calculate average speed by route and hour
route_speed = (
    df_non_zero
    .groupby(["PULocationID", "DOLocationID", "pickup_hour"])
    .agg(
        avg_distance=("trip_distance", "mean"),
        avg_duration=("trip_duration_hours", "mean")
    )
    .reset_index()
)

# Speed = distance / duration
route_speed["avg_speed_mph"] = (
    route_speed["avg_distance"] / route_speed["avg_duration"]
)

route_speed.head()

In [ ]:
# Filter meaningful routes
route_speed_filtered = route_speed[
    (route_speed["avg_distance"] > 1) &
    (route_speed["avg_speed_mph"] > 0)
]

# Find slowest routes
slowest_routes = (
    route_speed_filtered
    .sort_values("avg_speed_mph")
    .head(10)
)

slowest_routes

In [ ]:
# Merge pickup zone names
slowest_routes = slowest_routes.merge(
    zones[["LocationID", "zone", "borough"]],
    left_on="PULocationID",
    right_on="LocationID",
    how="left"
).rename(columns={"zone": "pickup_zone", "borough": "pickup_borough"})

# Merge drop-off zone names
slowest_routes = slowest_routes.merge(
    zones[["LocationID", "zone", "borough"]],
    left_on="DOLocationID",
    right_on="LocationID",
    how="left"
).rename(columns={"zone": "dropoff_zone", "borough": "dropoff_borough"})

slowest_routes[
    ["pickup_zone", "dropoff_zone", "pickup_hour", "avg_speed_mph"]
]

How does identifying high-traffic, high-demand routes help us?

Answer: Operational Insight: High-Traffic & High-Demand Routes

Identifying high-traffic and high-demand routes helps improve both operational efficiency and customer experience.

From the analysis, we observed that certain pickup–dropoff zone pairs consistently show very low average speeds during specific hours, indicating congestion and bottlenecks.

**Key benefits of identifying these routes:**

1. Better driver allocation
Taxi fleets can proactively position more vehicles in high-demand zones during peak hours to reduce passenger waiting times.

2. Congestion management
Routes with extremely low average speeds highlight areas affected by traffic congestion. These insights can be used to suggest alternative routing strategies or avoid inefficient routes during peak times.

3. Improved trip time estimates
Understanding time-of-day speed variations allows more accurate ETA predictions, improving customer satisfaction.

4. Dynamic pricing and incentives
High-demand, slow-moving routes may justify surge pricing or driver incentives to balance supply and demand.

5. Urban planning and policy support
Persistently slow routes can indicate infrastructure bottlenecks, supporting data-driven decisions for traffic management and city planning.

**Conclusion**

By analysing route-level speeds across different hours, taxi operators can optimize fleet deployment, reduce delays, and increase overall system efficiency, while passengers benefit from more reliable and predictable journeys.

**3.2.2** <font color = red>[3 marks]</font> <br>
Calculate the number of trips at each hour of the day and visualise them. Find the busiest hour and show the number of trips for that hour.

In [ ]:
# Visualise the number of trips per hour and find the busiest hour
# Count trips per hour (sampled data)
hourly_trips_sampled = df_non_zero["pickup_hour"].value_counts().sort_index()

# Scale up using sampling ratio
# (adjust this if your sampling ratio is different)
sampling_ratio = 0.1  # e.g. 10% sample
hourly_trips_actual = hourly_trips_sampled / sampling_ratio

hourly_trips_actual

In [ ]:
plt.figure(figsize=(10, 5))
hourly_trips_actual.plot(kind="bar")
plt.xlabel("Hour of Day")
plt.ylabel("Estimated Number of Trips")
plt.title("Estimated Taxi Trips per Hour (Scaled)")
plt.grid(axis="y")
plt.show()

In [ ]:
# Find busiest hour
busiest_hour = hourly_trips_actual.idxmax()
busiest_trips = hourly_trips_actual.max()

busiest_hour, busiest_trips

Remember, we took a fraction of trips. To find the actual number, you have to scale the number up by the sampling ratio.

**3.2.3** <font color = red>[2 mark]</font> <br>
Find the actual number of trips in the five busiest hours

In [ ]:
# Scale up the number of trips

# Fill in the value of your sampling fraction and use that to scale up the numbers
sample_fraction = 0.10

In [ ]:
# Number of trips per hour (sampled)
hourly_trips_sampled = df_non_zero["pickup_hour"].value_counts().sort_index()

In [ ]:
# Scale up to actual estimated trips
hourly_trips_actual = hourly_trips_sampled / sample_fraction

In [ ]:
# Find the five busiest hours
top_5_hours = hourly_trips_actual.sort_values(ascending=False).head(5)
top_5_hours

**3.2.4** <font color = red>[3 marks]</font> <br>
Compare hourly traffic pattern on weekdays. Also compare for weekend.

In [ ]:
# Compare traffic trends for the week days and weekends
# Sampling fraction used earlier
sample_fraction = 0.10

# Create day type: Weekday vs Weekend
df_non_zero["day_type"] = df_non_zero["tpep_pickup_datetime"].dt.weekday.apply(
    lambda x: "Weekend" if x >= 5 else "Weekday"
)

# Hourly trips by day type (sampled)
hourly_daytype = (
    df_non_zero
    .groupby(["day_type", "pickup_hour"])
    .size()
    .unstack(0)
    .fillna(0)
)

# Scale to actual estimated trips
hourly_daytype_actual = hourly_daytype / sample_fraction

# Plot comparison
plt.figure(figsize=(10, 5))
hourly_daytype_actual["Weekday"].plot(label="Weekday", marker="o")
hourly_daytype_actual["Weekend"].plot(label="Weekend", marker="o")

plt.xlabel("Hour of Day")
plt.ylabel("Estimated Number of Trips")
plt.title("Hourly Taxi Traffic: Weekdays vs Weekends (Scaled)")
plt.legend()
plt.grid(True)
plt.show()

hourly_daytype_actual

What can you infer from the above patterns? How will finding busy and quiet hours for each day help us?

Answer:

Inference from the patterns

From the weekday vs weekend hourly traffic analysis, we observe clear and consistent differences in travel behaviour:

 - Weekdays
1. Taxi demand rises sharply from early morning (around 7–9 AM), corresponding to office commute hours.
2. Demand peaks in the late afternoon to early evening (5–7 PM), indicating evening commute and post-work travel.
3. There is a noticeable drop in demand late at night.

 - Weekends

1. Morning demand is significantly lower compared to weekdays.
2. Demand increases gradually from late morning and remains relatively steady through the afternoon and evening.
3. Peak hours are later in the day compared to weekdays, reflecting leisure-oriented travel rather than work commutes.
improved customer experience, and smarter pricing and resource allocation.

Overall, weekdays show strong commute-driven peaks, while weekends show flatter, leisure-driven demand patterns.

**How identifying busy and quiet hours helps**

Identifying busy and quiet hours provides several operational and strategic benefits:

1. Fleet optimisation
2. Reduced passenger wait times
3. Dynamic pricing and incentives
4. Driver shift planning
5. Traffic and city planning insights

**3.2.5** <font color = red>[3 marks]</font> <br>
Identify top 10 zones with high hourly pickups. Do the same for hourly dropoffs. Show pickup and dropoff trends in these zones.

In [ ]:
# Find top 10 pickup and dropoff zones
# Find top 10 pickup zones
top_pickup_zones = (
    df_non_zero.groupby("PULocationID")
    .size()
    .sort_values(ascending=False)
    .head(10)
)

# Find top 10 dropoff zones
top_dropoff_zones = (
    df_non_zero.groupby("DOLocationID")
    .size()
    .sort_values(ascending=False)
    .head(10)
)

# Hourly pickup trends for top pickup zones
pickup_trends = (
    df_non_zero[df_non_zero["PULocationID"].isin(top_pickup_zones.index)]
    .groupby(["pickup_hour", "PULocationID"])
    .size()
    .unstack(fill_value=0)
    / sample_fraction
)

# Hourly dropoff trends for top dropoff zones
dropoff_trends = (
    df_non_zero[df_non_zero["DOLocationID"].isin(top_dropoff_zones.index)]
    .groupby(["pickup_hour", "DOLocationID"])
    .size()
    .unstack(fill_value=0)
    / sample_fraction
)

pickup_trends.head(), dropoff_trends.head()

**3.2.6** <font color = red>[3 marks]</font> <br>
Find the ratio of pickups and dropoffs in each zone. Display the 10 highest (pickup/drop) and 10 lowest (pickup/drop) ratios.

In [ ]:
# Find the top 10 and bottom 10 pickup/dropoff ratios
# Pickup counts per zone
pickup_counts = df_non_zero.groupby("PULocationID").size()

# Dropoff counts per zone
dropoff_counts = df_non_zero.groupby("DOLocationID").size()

# Combine into one DataFrame
pickup_dropoff_ratio = (
    pd.DataFrame({
        "pickups": pickup_counts,
        "dropoffs": dropoff_counts
    })
    .fillna(0)
)

# Avoid division by zero
pickup_dropoff_ratio = pickup_dropoff_ratio[pickup_dropoff_ratio["dropoffs"] > 0]

# Compute ratio
pickup_dropoff_ratio["pickup_dropoff_ratio"] = (
    pickup_dropoff_ratio["pickups"] / pickup_dropoff_ratio["dropoffs"]
)

# Top 10 highest ratios
top_10_highest = pickup_dropoff_ratio.sort_values(
    "pickup_dropoff_ratio", ascending=False
).head(10)

# Top 10 lowest ratios
top_10_lowest = pickup_dropoff_ratio.sort_values(
    "pickup_dropoff_ratio", ascending=True
).head(10)

top_10_highest, top_10_lowest

**3.2.7** <font color = red>[3 marks]</font> <br>
Identify zones with high pickup and dropoff traffic during night hours (11PM to 5AM)

In [ ]:
# During night hours (11pm to 5am) find the top 10 pickup and dropoff zones
# Note that the top zones should be of night hours and not the overall top zones
# Define night hours (11 PM to 5 AM)
night_hours = list(range(23, 24)) + list(range(0, 6))

night_trips = df_non_zero[df_non_zero["pickup_hour"].isin(night_hours)]

# Top 10 pickup zones at night
top_night_pickups = (
    night_trips.groupby("PULocationID")
    .size()
    .sort_values(ascending=False)
    .head(10)
    / sample_fraction
)

# Top 10 dropoff zones at night
top_night_dropoffs = (
    night_trips.groupby("DOLocationID")
    .size()
    .sort_values(ascending=False)
    .head(10)
    / sample_fraction
)

top_night_pickups, top_night_dropoffs


Now, let us find the revenue share for the night time hours and the day time hours. After this, we will move to deciding a pricing strategy.

**3.2.8** <font color = red>[2 marks]</font> <br>
Find the revenue share for nighttime and daytime hours.

In [ ]:
# Filter for night hours (11 PM to 5 AM)
night_hours = (df_non_zero["pickup_hour"] >= 23) | (df_non_zero["pickup_hour"] <= 5)

# Calculate total revenue
total_revenue = df_non_zero["total_amount"].sum()

# Nighttime revenue
night_revenue = df_non_zero.loc[night_hours, "total_amount"].sum()

# Daytime revenue
day_revenue = total_revenue - night_revenue

# Revenue share
revenue_share = pd.Series(
    {
        "Nighttime Revenue Share": night_revenue / total_revenue,
        "Daytime Revenue Share": day_revenue / total_revenue
    }
)

revenue_share

##### Pricing Strategy

**3.2.9** <font color = red>[2 marks]</font> <br>
For the different passenger counts, find the average fare per mile per passenger.

For instance, suppose the average fare per mile for trips with 3 passengers is 3 USD/mile, then the fare per mile per passenger will be 1 USD/mile.

In [ ]:
# Analyse the fare per mile per passenger for different passenger counts

# Avoid divide-by-zero trips
df_fare_mile = df_non_zero[df_non_zero["trip_distance"] > 0].copy()

# Fare per mile
df_fare_mile["fare_per_mile"] = df_fare_mile["fare_amount"] / df_fare_mile["trip_distance"]

# Fare per mile per passenger
df_fare_mile["fare_per_mile_per_passenger"] = (
    df_fare_mile["fare_per_mile"] / df_fare_mile["passenger_count"]
)

# Average fare per mile per passenger by passenger count
avg_fare_per_mile_per_passenger = (
    df_fare_mile
    .groupby("passenger_count")["fare_per_mile_per_passenger"]
    .mean()
)

avg_fare_per_mile_per_passenger

**3.2.10** <font color = red>[3 marks]</font> <br>
Find the average fare per mile by hours of the day and by days of the week

In [ ]:
# Compare the average fare per mile for different days and for different times of the day
# Reuse fare_per_mile dataframe (already excludes zero distance)
df_fpm = df_fare_mile.copy()

# Add pickup hour and day name
df_fpm["pickup_hour"] = df_fpm["tpep_pickup_datetime"].dt.hour
df_fpm["pickup_day"] = df_fpm["tpep_pickup_datetime"].dt.day_name()

# Average fare per mile by hour of day
avg_fpm_hour = (
    df_fpm
    .groupby("pickup_hour")["fare_per_mile"]
    .mean()
)

# Average fare per mile by day of week (ordered)
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
avg_fpm_day = (
    df_fpm
    .groupby("pickup_day")["fare_per_mile"]
    .mean()
    .reindex(day_order)
)

# Plot comparison
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
avg_fpm_hour.plot(marker="o")
plt.title("Average Fare per Mile by Hour of Day")
plt.xlabel("Hour of Day")
plt.ylabel("Fare per Mile ($)")
plt.grid(True)

plt.subplot(1, 2, 2)
avg_fpm_day.plot(marker="o")
plt.title("Average Fare per Mile by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Fare per Mile ($)")
plt.grid(True)

plt.tight_layout()
plt.show()

avg_fpm_hour, avg_fpm_day

**3.2.11** <font color = red>[3 marks]</font> <br>
Analyse the average fare per mile for the different vendors for different hours of the day

In [ ]:
df.columns

In [ ]:
# Compare fare per mile for different vendors
# Ensure fare_per_mile exists and valid trips only
df_vendor_fpm = df_non_zero[df_non_zero["trip_distance"] > 0].copy()
df_vendor_fpm["fare_per_mile"] = (
    df_vendor_fpm["fare_amount"] / df_vendor_fpm["trip_distance"]
)

# Extract pickup hour
df_vendor_fpm["pickup_hour"] = df_vendor_fpm["tpep_pickup_datetime"].dt.hour

# Average fare per mile by vendor and hour
vendor_hour_fpm = (
    df_vendor_fpm
    .groupby(["VendorID", "pickup_hour"])["fare_per_mile"]
    .mean()
    .unstack(0)
)

# Plot
plt.figure(figsize=(12, 5))
vendor_hour_fpm.plot(marker="o")

plt.xlabel("Hour of Day")
plt.ylabel("Average Fare per Mile ($)")
plt.title("Average Fare per Mile by Vendor and Hour of Day")
plt.grid(True)
plt.legend(title="VendorID")
plt.show()

vendor_hour_fpm

**3.2.12** <font color = red>[5 marks]</font> <br>
Compare the fare rates of the different vendors in a tiered fashion. Analyse the average fare per mile for distances upto 2 miles. Analyse the fare per mile for distances from 2 to 5 miles. And then for distances more than 5 miles.


In [ ]:
# Defining distance tiers
# Keep valid trips only
df_tier = df[
    (df["trip_distance"] > 0) &
    (df["fare_amount"] > 0)
].copy()

# Fare per mile
df_tier["fare_per_mile"] = df_tier["fare_amount"] / df_tier["trip_distance"]

# Define distance tiers
df_tier["distance_tier"] = pd.cut(
    df_tier["trip_distance"],
    bins=[0, 2, 5, float("inf")],
    labels=["0–2 miles", "2–5 miles", "5+ miles"]
)

# Average fare per mile by vendor and distance tier
vendor_distance_fpm = (
    df_tier
    .groupby(["VendorID", "distance_tier"])["fare_per_mile"]
    .mean()
    .unstack()
)

vendor_distance_fpm

##### Customer Experience and Other Factors

**3.2.13** <font color = red>[5 marks]</font> <br>
Analyse average tip percentages based on trip distances, passenger counts and time of pickup. What factors lead to low tip percentages?

In [ ]:
#  Analyze tip percentages based on distances, passenger counts and pickup times
# Keep valid trips
df_tip = df[
    (df["fare_amount"] > 0) &
    (df["trip_distance"] > 0)
].copy()

# Tip percentage
df_tip["tip_pct"] = (df_tip["tip_amount"] / df_tip["fare_amount"]) * 100

# Pickup hour
df_tip["pickup_hour"] = df_tip["tpep_pickup_datetime"].dt.hour

# Distance tiers
df_tip["distance_tier"] = pd.cut(
    df_tip["trip_distance"],
    bins=[0, 2, 5, float("inf")],
    labels=["0–2 miles", "2–5 miles", "5+ miles"]
)

# Average tip % by distance, passengers, and hour
tip_summary = (
    df_tip
    .groupby(["distance_tier", "passenger_count", "pickup_hour"])["tip_pct"]
    .mean()
    .reset_index()
)

tip_summary.head()

Additional analysis [optional]: Let's try comparing cases of low tips with cases of high tips to find out if we find a clear aspect that drives up the tipping behaviours

In [ ]:
# Compare trips with tip percentage < 10% to trips with tip percentage > 25%
low_tips = df_tip[df_tip["tip_pct"] < 10]
high_tips = df_tip[df_tip["tip_pct"] > 25]

comparison = pd.DataFrame({
    "Low Tip Trips (<10%)": low_tips[["trip_distance", "passenger_count", "pickup_hour"]].mean(),
    "High Tip Trips (>25%)": high_tips[["trip_distance", "passenger_count", "pickup_hour"]].mean()
})

comparison

**3.2.14** <font color = red>[3 marks]</font> <br>
Analyse the variation of passenger count across hours and days of the week.

In [ ]:
# See how passenger count varies across hours and days
# Create pickup hour and pickup day
df_pc = df.copy()
df_pc["pickup_hour"] = df_pc["tpep_pickup_datetime"].dt.hour
df_pc["pickup_day"] = df_pc["tpep_pickup_datetime"].dt.day_name()

# Average passenger count by hour
avg_passenger_hour = (
    df_pc
    .groupby("pickup_hour")["passenger_count"]
    .mean()
)

# Average passenger count by day
avg_passenger_day = (
    df_pc
    .groupby("pickup_day")["passenger_count"]
    .mean()
    .reindex(["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"])
)

# Plot
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
avg_passenger_hour.plot(marker="o")
plt.title("Average Passenger Count by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Avg Passenger Count")
plt.grid(True)

plt.subplot(1,2,2)
avg_passenger_day.plot(marker="o")
plt.title("Average Passenger Count by Day")
plt.xlabel("Day of Week")
plt.ylabel("Avg Passenger Count")
plt.grid(True)

plt.tight_layout()
plt.show()

avg_passenger_hour, avg_passenger_day


**3.2.15** <font color = red>[2 marks]</font> <br>
Analyse the variation of passenger counts across zones

In [ ]:
# How does passenger count vary across zones
passenger_by_zone = (
    df
    .groupby("PULocationID")["passenger_count"]
    .mean()
    .sort_values(ascending=False)
)

passenger_by_zone

In [ ]:
zones_with_trips.columns

In [ ]:
# For a more detailed analysis, we can use the zones_with_trips GeoDataFrame
# Create a new column for the average passenger count in each zone.
avg_passenger_by_zone = (
    df
    .groupby("PULocationID")["passenger_count"]
    .mean()
    .reset_index()
)

zones_with_trips = zones_with_trips.merge(
    avg_passenger_by_zone,
    on="PULocationID",
    how="left"
)

zones_with_trips.rename(columns={"passenger_count": "avg_passenger_count"}, inplace=True)

zones_with_trips[["zone", "avg_passenger_count"]].drop_duplicates()

Find out how often surcharges/extra charges are applied to understand their prevalance

**3.2.16** <font color = red>[5 marks]</font> <br>
Analyse the pickup/dropoff zones or times when extra charges are applied more frequently

In [ ]:
# How often is each surcharge applied?
surcharge_cols = [
    "extra",
    "mta_tax",
    "tolls_amount",
    "congestion_surcharge",
    "airport_fee"
]

In [ ]:
 #Flag trips with any surcharge applied
df["any_surcharge"] = df[surcharge_cols].gt(0).any(axis=1)

In [ ]:
# Surcharge frequency by pickup zone
surcharge_by_zone = (
    df
    .groupby("PULocationID")["any_surcharge"]
    .mean()
    .sort_values(ascending=False)
)

surcharge_by_zone

## **4** Conclusion
<font color = red>[15 marks]</font> <br>

### **4.1** Final Insights and Recommendations
<font color = red>[15 marks]</font> <br>

Conclude your analyses here. Include all the outcomes you found based on the analysis.

Based on the insights, frame a concluding story explaining suitable parameters such as location, time of the day, day of the week etc. to be kept in mind while devising a strategy to meet customer demand and optimise supply.

Answer: Based on the exploratory data analysis of the 2023 NYC Yellow Taxi dataset, several clear patterns emerge across time, location, pricing, and passenger behavior. Taxi demand is strongly influenced by time of day, day of week, and location, while revenue and pricing are primarily driven by trip distance, congestion, and location-specific surcharges.

Pickup activity peaks during morning (7–10 AM) and evening (4–8 PM) hours, aligning with commuter travel patterns. Weekdays consistently show higher demand than weekends, with Friday being the busiest day overall. Monthly trends indicate relatively stable demand throughout the year, with moderate increases during spring and fall, reflecting seasonal mobility patterns in New York City.

Financial analysis shows a strong positive correlation between trip distance and fare amount, confirming distance-based pricing as the dominant revenue driver. Fare per mile decreases as trip distance increases, indicating the presence of base fares that disproportionately affect short trips. Passenger occupancy is largely dominated by single-passenger rides, though higher average passenger counts are observed in airport and commercial hub zones, suggesting more group travel in these areas.

Surcharges and extra charges are not uniformly distributed. Congestion surcharges, airport fees, and tolls are applied more frequently in central business districts, airports, and during peak hours, reinforcing the impact of both location and time on final trip cost.

Overall, the analysis highlights opportunities to optimize routing, fleet positioning, and pricing strategies by leveraging predictable demand and surcharge patterns observed in the data

**4.1.1** <font color = red>[5 marks]</font> <br>
Recommendations to optimize routing and dispatching based on demand patterns and operational inefficiencies

Taxi routing and dispatch efficiency can be significantly improved by aligning vehicle availability with observed demand patterns. Since demand peaks during weekday commuting hours, fleet dispatch should be intensified during morning and evening peak periods, particularly in high-density business zones and transit hubs. During late-night and early-morning hours, demand drops substantially, allowing for reduced active fleet size and optimized driver shifts.

Congestion-heavy areas should be served using dynamic routing strategies that minimize idle time and avoid bottlenecks, especially during peak hours when congestion surcharges are prevalent. Incorporating historical pickup density and time-based demand forecasts into dispatch systems can reduce empty cruising, lower fuel costs, and improve trip turnaround times.

**4.1.2** <font color = red>[5 marks]</font> <br>

Suggestions on strategically positioning cabs across different zones to make best use of insights uncovered by analysing trip trends across time, days and months.

Strategic cab positioning should prioritize zones with consistently high pickup volumes and higher passenger occupancy. Airport zones, commercial centers, and central Manhattan locations show higher trip frequency and passenger counts, making them ideal for proactive vehicle staging.

Conversely, zones with persistently low demand should rely on on-demand relocation rather than permanent fleet presence. Seasonal and monthly trends also suggest repositioning more vehicles during high-demand months and scaling back during lower-demand periods. This spatial rebalancing ensures better asset utilization while maintaining adequate service coverage across the city.

**4.1.3** <font color = red>[5 marks]</font> <br>
Propose data-driven adjustments to the pricing strategy to maximize revenue while maintaining competitive rates with other vendors.

Pricing analysis reveals that short trips are disproportionately affected by base fares, while longer trips benefit from lower average fare per mile. To remain competitive, pricing strategies can be refined by moderating base fare impact during off-peak hours or offering incentives for short trips in low-demand periods.

Given the high prevalence of surcharges in congestion zones and airports, transparent communication of surcharge components can improve customer satisfaction. Additionally, targeted promotions during low-demand hours or in underutilized zones can stimulate demand without undermining revenue. Leveraging distance-based elasticity insights allows vendors to balance competitiveness with profitability while maintaining consistency across vendors.

## Assumptions

- A 5% hourly stratified sample adequately represents overall taxi demand patterns.
- Zero fare and zero total amount trips represent cancelled or erroneous trips and were excluded from financial analysis.
- Zero tip amounts are valid and retained.
- Short-distance trips with zero distance but identical pickup and dropoff zones are valid.
- Missing passenger counts were imputed using the median to preserve realistic occupancy patterns.
- Negative monetary values are data errors and were capped at zero.
